# TTS와 STT

`TTS(Text-to-Speech)`는 텍스트를 음성으로 변환하는 기능이다.

TTS는 입력 텍스트를 사람이 들을 수 있는 mp3, wav 같은 음성 파일로 변환한다.

`STT(Speech-to-Text)`는 음성을 텍스트로 변환하는 기능이다.

사용자가 마이크로 말한 음성 파일을 STT에 전달하면, 모델은 음성 내용을 인식하여 텍스트로 반환한다.

OpenAI Audio API에서는 텍스트를 음성으로 바꾸는 speech 기능과, 음성을 텍스트로 바꾸는 transcription 기능을 제공한다.

공식 문서:
- Text to speech: https://developers.openai.com/api/docs/guides/text-to-speech
- Speech to text: https://developers.openai.com/api/docs/guides/speech-to-text

## 모델 구분

| 구분 | 역할 | 예시 모델 |
|---|---|---|
| 텍스트 모델 | 안내문 생성, 요약, 답변 생성 | gpt-4.1-mini |
| TTS 모델 | 텍스트를 음성으로 변환 | gpt-4o-mini-tts, tts-1, tts-1-hd |
| STT 모델 | 음성을 텍스트로 변환 | gpt-4o-mini-transcribe, gpt-4o-transcribe, whisper-1 |

## 음성 스타일

TTS에서 음성 스타일은 크게 두 가지로 조절한다.

1. voice
   - 음색 자체를 선택한다.
   - 예: alloy, ash, ballad, coral, echo, fable, onyx, nova, sage, shimmer, verse, marin, cedar

2. instructions
   - 같은 voice라도 말투, 분위기, 감정, 속도 등을 지시할 수 있다.
   - 예: 뉴스 앵커처럼 차분하게, 고객센터 상담원처럼 친절하게, 초등학생에게 설명하듯 쉽게

In [1]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델명을 읽어온다.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPEAI_API_KEY를 환경 변수로 설정하세요.")

client = OpenAI(api_key=api_key)

# 텍스트 모델
TEXT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-4.1-mini")

# TTS/STT 모델
TTS_MODEL = os.getenv("OPENAI_TTS_MODEL", "gpt-4o-mini-tts")
STT_MODEL = os.getenv("OPENAI_STT_MODEL", "gpt-4o-mini-transcribe")

print("OpenAI client 준비 완료")
print("TEXT 모델 : ", TEXT_MODEL)
print("TTS 모델 : ", TTS_MODEL)
print("STT 모델 : ", STT_MODEL)

OpenAI client 준비 완료
TEXT 모델 :  gpt-4.1-mini
TTS 모델 :  gpt-4o-mini-tts
STT 모델 :  gpt-4o-mini-transcribe


## 파일 경로 설정

In [2]:
from pathlib import Path

AUDIO_DIR = Path("audio_outputs")
AUDIO_DIR.mkdir(exist_ok=True)

## LLM으로 음성 안내문 만들기

In [3]:
response = client.responses.create(
    model=TEXT_MODEL,
    instructions="너는 AI 수업 진행을 돕는 강사다. 안내 멘트는 자연스럽고 간결하게 작성한다.",
    input="TTS, STT 실습을 시작하기 전 수강생에게 들려줄 4문장 안내 멘트를 작성해줘"
)

script = response.output_text 
print(script)

안녕하세요, 여러분! 오늘은 TTS와 STT 기술을 직접 체험해보는 시간을 가질 예정입니다. 차근차근 따라 하시면서 음성 인식과 음성 합성 기능을 익혀보세요. 궁금한 점이 있으면 언제든지 질문해 주세요.


## TTS: 텍스트를 음성으로 변환
`audio.speech` 기능을 사용하면 텍스트를 음성 파일로 변환할 수 있다.

In [4]:
from IPython.display import Audio, display

# 저장할 음성 파일 경로
tts_path = AUDIO_DIR / "class_announcement.mp3"

# TTS 음색 선택
voice = "nova"

with client.audio.speech.with_streaming_response.create(
    model=TTS_MODEL,
    voice=voice,
    input=script
) as response:
    response.stream_to_file(tts_path)

print("음성 파일 저장 완료:", tts_path)

# 주피터 노트북에서 mp3 파일 재생
display(Audio(str(tts_path)))

음성 파일 저장 완료: audio_outputs\class_announcement.mp3


In [8]:
# 테스트할 말투 목록
script_styles = [
    {"name" : "뉴스 앵커", "file" : "news_anchor"},
    {"name" : "초등학생 대상 안내", "file" : "elementary"},
    {"name" : "스포츠 중계 캐스터", "file" : "sports_caster"}
]

script_style = script_styles[2]

response = client.responses.create(
    model=TEXT_MODEL,
    instructions=f"너는 {script_style['name']} 말투로 짧은 안내 문장을 작성한다.",
    input="어버이날에 대한 설명을 2문장으로 작성해줘."
)

tone_script = response.output_text 
print("선택한 말투 :", script_style['name'])
print(tone_script)

# 저장할 음성 파일 경로
tts_path = AUDIO_DIR / f"tone_practice_{script_style['file']}.mp3"

with client.audio.speech.with_streaming_response.create(
    model=TTS_MODEL,
    voice=voice,
    input=tone_script
) as response:
    response.stream_to_file(tts_path)

print("음성 파일 저장 완료:", tts_path)

# 주피터 노트북에서 mp3 파일 재생
display(Audio(str(tts_path)))

선택한 말투 : 스포츠 중계 캐스터
자, 여러분! 어버이날은 부모님의 사랑과 헌신에 감사하는 특별한 날입니다! 오늘은 마음 담아 따뜻한 감사의 인사를 전해보시죠!
음성 파일 저장 완료: audio_outputs\tone_practice_sports_caster.mp3


## STT: 음성을 텍스트로 변환
`audio.transcriptions` 기능을 사용하면 음성 파일을 텍스트로 변환할 수 있다.

In [9]:
# 직전에 생성한 음성 파일을 바이너리 읽기 모드로 연다.
# stt api에는 파일 객체를 전달해야 한다.
with open(tts_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model=STT_MODEL,
        file=audio_file
    )

# 변환 된 텍스트 확인
print(transcript.text)

자, 여러분, 어버이날은 부모님의 사랑과 헌신에 감사하는 특별한 날입니다. 오늘은 마음 담아 따뜻한 감사의 인사를 전해보시죠.


## 음성 기반 AI 앱의 기본 구조
실제 음성 기반 AI 앱에서는 TTS와 STT가 따로 떨어져 동작하지 않는다.
보통 STT -> LLM -> TTS 와 같이 하나의 흐름으로 연결 된다.

In [10]:
def voice_round_trip(audio_path, question_prefix="다음 음성 내용을 듣고 짧고 자연스럽게 응답해줘"):
    """음성 파일을 입력 받아 STT -> LLM -> TTS 순서로 처리한다."""

    # 음성 파일을 STT 모델에 전달해 텍스트로 변환
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model=STT_MODEL,
            file=audio_file
        )
    
    recognized_text = transcript.text

    # STT 결과를 LLM에 전달해 답변 생성
    answer = client.responses.create(
        model=TEXT_MODEL,
        instructions="너는 음성 기반 학습 도우미다. 짧고 명확하게 답한다.",
        input=f"""
{question_prefix}

[음성 인식 결과]
{recognized_text}
"""
    ).output_text

    # LLM이 생성한 답변을 다시 TTS로 변환
    output_path = AUDIO_DIR / "ai_voice_answer.mp3"

    with client.audio.speech.with_streaming_response.create(
        model=TTS_MODEL,
        voice=voice,
        input=answer
    ) as response:
        response.stream_to_file(output_path)

    # 중간 결과와 최종 음성 파일 경로를 반환
    return recognized_text, answer, output_path

## 직접 녹음한 음성 변환

In [11]:
# 필요한 경우 먼저 설치한다
# sounddevice : 마이크 입력을 Python에서 녹음하기 위한 라이브러리
# scipy : 녹음 된 데이터를 wav 파일로 저장하기 위한 라이브러리 
%pip install sounddevice scipy -q

Note: you may need to restart the kernel to use updated packages.


In [13]:
import sounddevice as sd 
from scipy.io.wavfile import write 
from IPython.display import Audio, display

# 녹음 시간 설정
duration = 10

# 샘플링 레이트 설정
sample_rate = 16000

record_path = AUDIO_DIR / "my_recording.wav"

print("녹음을 시작합니다. 5초동안 말해보세요.")

recoding = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="int16"
)

# 녹음이 끝날 때까지 코드 실행을 잠시 멈춘다.
sd.wait()

write(record_path, sample_rate, recoding)

print("녹음 파일 저장 완료: ", record_path)

display(Audio(str(record_path)))

녹음을 시작합니다. 5초동안 말해보세요.
녹음 파일 저장 완료:  audio_outputs\my_recording.wav


In [14]:
recognized_text, answer, output_path = voice_round_trip(
    record_path
)

print("[STT 결과]")
print(recognized_text)
print("[LLM 답변]")
print(answer)
print("[TTS 결과]")
display(Audio(str(output_path)))

[STT 결과]
OpenAI에서 TTS랑 STT 기능을 사용하기 위해서 어떻게 해야 되는지 가이드 해줄래?
[LLM 답변]
OpenAI API에서 TTS와 STT 기능을 사용하려면, 먼저 OpenAI 개발자 계정을 만들고 API 키를 발급받아야 해요. 그 후에, OpenAI의 음성 관련 엔드포인트 문서를 참고해 API를 호출하면 됩니다.
[TTS 결과]
